In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Dropout

In [2]:
import tensorflow_datasets as tfds
(ds_train, ds_test), ds_info = tfds.load('emnist/letters',split=['train', 'test'],shuffle_files=True,as_supervised=True,with_info=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.ZFSBPF_3.1.0/emnist-train.tfrecord*...:   0%|   …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.ZFSBPF_3.1.0/emnist-test.tfrecord*...:   0%|    …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/letters/3.1.0. Subsequent calls will reuse this data.


In [17]:
def prep(dataset):
  images=[]
  labels=[]
  for img,label in dataset:
   img=tf.cast(img,tf.float32)
   img=tf.reshape(img,(28,28,1))

   label=label-1
   labels.append(label)
   images.append(img)

  return tf.stack(images), tf.stack(labels)

In [21]:
x_train, y_train = prep(ds_train)
x_test, y_test = prep(ds_test)

In [22]:
"""from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, 26)
y_test = to_categorical(y_test, 26)"""

'from tensorflow.keras.utils import to_categorical\ny_train = to_categorical(y_train, 26)\ny_test = to_categorical(y_test, 26)'

In [23]:
##model architecture:
model=Sequential()
model.add(Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(26, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [24]:
## model compilation
model.compile(optimizer='Adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 26)             │         3,354 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 227,098 (887.10 KB)

 Trainable params: 227,098 (887.10 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
## training the model:
model.fit(x_train,y_train,epochs=5,batch_size=32,validation_data=(x_test, y_test))

Epoch 1/5
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.5274 - loss: 1.9432 - val_accuracy: 0.8603 - val_loss: 0.4096
Epoch 2/5
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.8498 - loss: 0.4618 - val_accuracy: 0.9020 - val_loss: 0.2966
Epoch 3/5
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8867 - loss: 0.3517 - val_accuracy: 0.9044 - val_loss: 0.2880
Epoch 4/5
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9043 - loss: 0.2930 - val_accuracy: 0.9124 - val_loss: 0.2553
Epoch 5/5
2775/2775 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9121 - loss: 0.2669 - val_accuracy: 0.9132 - val_loss: 0.2593


In [26]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_accuracy)
print("Loss:",test_loss)

463/463 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9116 - loss: 0.2577
Test Accuracy: 0.9132432341575623
Loss: 0.2592984139919281
